# IMPACT-Polish v2 - baseline Tesseract na zamrożonym teście

**Benchmark:** 36 stron polskiego druku historycznego (3 kolekcje), 52 291 znaków GT.
Test jest zamrożony (split odziedziczony z v1, per kolekcja, zero wycieku).

**Protokół metryk:** `training/benchmark_pages.py` - CER/WER micro po normalizacji
Unicode NFC + białe znaki; wielkość liter i diakrytyki zachowane. Strony z błędem
OCR liczą się jako pusta hipoteza (status: error/missing).

**Kolejność:** 1 -> 2 -> 3 -> 4 -> 5. Komórka 4 wymaga GPU/systemu z apt (Kaggle, Colab, Lightning).
Poza Tesseract dodaj kolejne systemy jako osobne komórki generujące predictions JSONL:
`{"id", "status": "ok"|"error", "text", "elapsed_seconds"}`.

In [ ]:
# Pobranie zamrozonego bundla benchmarku z HF (publiczny dataset)
import subprocess, sys, os
from pathlib import Path
for _w in ('/kaggle/working', '/teamspace/studios/this_studio', '/content', '/workspace'):
    if Path(_w).exists():
        WORKDIR = Path(_w)
        break
else:
    WORKDIR = Path.cwd()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
from huggingface_hub import hf_hub_download
BUNDLE = hf_hub_download('PiotrSty/impact-print-v2', 'impact-print-v2-test.tar.gz', repo_type='dataset')
print('Bundle:', BUNDLE)
import tarfile
bench = WORKDIR / 'impact-print-v2'
if not (bench / 'test_manifest.jsonl').exists():
    with tarfile.open(BUNDLE) as tar:
        tar.extractall(WORKDIR)
print('Rozpakowane do', bench)
n = sum(1 for _ in (bench / 'test_manifest.jsonl').open(encoding='utf-8'))
print(f'Test: {n} stron (oczekiwane 36)')

In [ ]:
# Sanity check manifestu (identyczny z training/benchmark_pages.py)
import json, hashlib
from pathlib import Path
bench = WORKDIR / 'impact-print-v2'
records = [json.loads(l) for l in (bench / 'test_manifest.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
assert len({r['id'] for r in records}) == len(records), 'Duplicate ids'
for r in records:
    data = (bench / r['image']).read_bytes()
    assert hashlib.sha256(data).hexdigest() == r['sha256'], f"Checksum mismatch: {r['id']}"
total_chars = sum(len(r['text']) for r in records)
print(f'OK: {len(records)} stron, {total_chars} znakow GT, checksumy zgodne')

In [ ]:
# Instalacja Tesseract + polski model jezykowy
import subprocess, shutil
if not shutil.which('tesseract'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'tesseract-ocr', 'tesseract-ocr-pol'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytesseract', 'Pillow'], check=True)
r = subprocess.run(['tesseract', '--list-langs'], capture_output=True, text=True)
print(r.stdout)
assert 'pol' in r.stdout, 'Brak modelu pol - doinstaluj tesseract-ocr-pol'

In [ ]:
# Baseline: Tesseract pol, pelna strona
import json, time, warnings
from pathlib import Path
from PIL import Image
import pytesseract
warnings.filterwarnings('ignore')
bench = WORKDIR / 'impact-print-v2'
records = [json.loads(l) for l in (bench / 'test_manifest.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
preds_path = bench / 'tesseract_pol_preds.jsonl'
with preds_path.open('w', encoding='utf-8') as fh:
    for i, r in enumerate(records, 1):
        t0 = time.time()
        try:
            img = Image.open(bench / r['image'])
            text = pytesseract.image_to_string(img, lang='pol')
            status = 'ok'
        except Exception as e:
            text, status = '', 'error'
            print(f"{r['id']}: ERROR {e}")
        fh.write(json.dumps({'id': r['id'], 'status': status, 'text': text,
                             'elapsed_seconds': round(time.time() - t0, 2)}, ensure_ascii=False) + '\n')
        if i % 10 == 0:
            print(f'  {i}/{len(records)}', flush=True)
print('Predictions:', preds_path)

In [ ]:
# Ewaluacja offline (training/benchmark_pages.py, wbudowany w repo)
import json, sys, subprocess
from pathlib import Path
repo = WORKDIR / 'OCR_engine'
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/PiotrStyla/OCR_engine.git', str(repo)], check=True)
bench = WORKDIR / 'impact-print-v2'
r = subprocess.run([sys.executable, '-m', 'training.benchmark_pages',
                    '--manifest', str(bench / 'test_manifest.jsonl'),
                    '--predictions', str(bench / 'tesseract_pol_preds.jsonl'),
                    '--output', str(bench / 'tesseract_pol_v2.json')],
                   cwd=str(repo), capture_output=True, text=True)
print(r.stdout[-2000:] or r.stderr[-2000:])
print('=== ZAPISZ tesseract_pol_v2.json do repo (benchmarks/impact-print-v2/) ===')